# Agents Benchmark

<h3 style="text-align: center;">Table of Contents</h3>
<ol style="display: inline-block; text-align: left;">
  <li><a href="#Loops-/-Models-Scorecard">Loops / Models Scorecard</a></li>
  <li><a href="#Loop-2-by-2-Comparison">Loop 2-by-2 Comparison</a></li>
  <li><a href="#Per-Agent-Performance">Per-Agent Performance</a></li>
  <li><a href="#LLM-Cost-per-Agent">LLM Cost per Agent</a></li>
</ol>

In [ ]:
import os

import agents_gym_core.reporting as rp
from agents_gym_core.reporting.run_report import DEFAULT_RESULTS_PATHS

df = rp.load_data(DEFAULT_RESULTS_PATHS)
agg = os.getenv("AGGREGATION_MODE", "datapoint_run")  # the run's --aggregation_mode
confidence_level = float(os.getenv("CONFIDENCE_LEVEL", "0.95"))  # run_report --confidence_level
max_runs_to_plot = int(os.getenv("MAX_RUNS_TO_PLOT", "-1"))  # run_report --max_runs_to_plot
summary, per_run = rp.calculate_agent_scores_for_report(
    df, agg=agg, ci_confidence_level=confidence_level, max_runs_to_plot=max_runs_to_plot
)

<hr>

## Loops / Models Scorecard



**Reading the scorecard**

- ANOVA tests whether the loop/model group means are equal. The F statistic compares between-group and within-group mean squares; the p-value is the probability of an F at least this large under that null hypothesis.
- Boxes show hierarchical bootstrap confidence intervals at `confidence_level` (95% by default). White lines show mean scores, whiskers show observed minima and maxima, and points show individual runs.

[View paired loop/model comparisons](#Loop-2-by-2-Comparison).

In [ ]:
rp.run_loop_analysis(
    per_run,
    p_value_threshold_anova=0.05,
    p_value_threshold_tukey=0.05,
    ci_confidence_level=confidence_level,
    show_df=False,
)

<hr>

## Loop 2-by-2 Comparison



**Reading paired comparisons**

- Configurations are paired by `(agent, agent version, run)`. The paired t-test tests whether the mean score difference is zero.
- The violin shows `loop1 - loop2` score differences; its inner box shows quartiles. Positive values favor loop1, and negative values favor loop2.
- Dotted lines mark differences of ±0.1 (10 percentage points). The displayed p-value tests the mean paired difference, and the bootstrap confidence interval summarizes its uncertainty.

[View per-agent performance](#Per-Agent-Performance).

In [ ]:
rp.plot_violin_distribution_between_loops(per_run, p_value_threshold=0.05)

<hr>

## Per-Agent Performance


**Reading per-agent plots**

Bars show mean scores. Boxes show bootstrap confidence intervals across repeated runs at `confidence_level` (95% by default), and whiskers show observed minima and maxima. Use the paired comparisons above to test differences between configurations.

In [ ]:
rp.compare_loops_and_agent_versions(
    summary,
    per_run,
    clustering="best",
    orientation="best",
    use_ci_boxplots=True,
    ci_confidence_level=confidence_level,
)

<hr>

## LLM Cost per Agent

Per-agent LLM cost averaged across runs (sum across datapoints within a run, then mean across runs). Source: `final_score_per_agent.json`, aggregated from each run's `cost.json`. Empty when no runs recorded usage.


In [ ]:
_cost_df = rp.display_cost_table(DEFAULT_RESULTS_PATHS)